# Week 4 — Data Preprocessing & Feature Engineering
### Datasets: German Credit · LendingClub · Credit Card Fraud Detection

**Pipeline per dataset:**
1. Load from Drive (reuses Week 3 path detection)
2. Handle missing values
3. Engineer domain features (DTI, utilisation, employment stability)
4. Encode categoricals · scale numerics
5. Stratified train/val/test split preserving protected-group proportions
6. Apply SMOTE to **train only** (never val/test)
7. Save processed splits to Drive
8. Log everything to MLflow

**Environment setup (bottom of notebook):** MLflow tracking + Dockerfile + requirements.txt

> Run cells top-to-bottom. Cell 1 mounts Drive, Cell 2 installs deps, Cell 3 detects paths.


## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive mounted!")

Mounted at /content/drive
Drive mounted!


## Cell 2 — Install Dependencies

Installs `imbalanced-learn` (SMOTE), `mlflow`, and the `kaggle` CLI.

In [2]:
!pip install -q imbalanced-learn==0.12.3 mlflow==2.16.2 kaggle==1.6.17
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.2 MB/s eta 0:00:00
Depe

## Cell 3 — Set Up Kaggle API & Output Folder

**One-time setup:** you'll be asked to upload your `kaggle.json` token.

To get one: go to [kaggle.com](https://kaggle.com) → your profile icon → Settings → API section → **Create New Token**. A file named `kaggle.json` downloads. Upload it when prompted below.

If you've already uploaded it this session, re-running this cell is fine — it just reconfigures.

In [3]:
import os, json
from google.colab import files

# Upload kaggle.json if not already present
kaggle_dir = os.path.expanduser('~/.kaggle')
kaggle_json = f'{kaggle_dir}/kaggle.json'

if not os.path.exists(kaggle_json):
    print("Upload your kaggle.json file (from kaggle.com -> Settings -> Create New Token):")
    uploaded = files.upload()
    assert 'kaggle.json' in uploaded, "You need to upload kaggle.json"
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(kaggle_json, 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(kaggle_json, 0o600)
    print("kaggle.json installed at ~/.kaggle/kaggle.json")
else:
    print("kaggle.json already configured.")

# Output directory on Drive for processed splits & MLflow
OUTPUT_ROOT = '/content/drive/MyDrive/CreditRisk_Wk4'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(f'{OUTPUT_ROOT}/processed', exist_ok=True)
os.makedirs(f'{OUTPUT_ROOT}/mlruns', exist_ok=True)
print(f"Output root: {OUTPUT_ROOT}")

# Local workspace on Colab for downloaded raw files (fast; ephemeral)
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Raw data dir (Colab-local): {DATA_DIR}")

Upload your kaggle.json file (from kaggle.com -> Settings -> Create New Token):


Saving kaggle.json to kaggle.json
kaggle.json installed at ~/.kaggle/kaggle.json
Output root: /content/drive/MyDrive/CreditRisk_Wk4
Raw data dir (Colab-local): /content/data


## Cell 3b — Download Datasets

- **German Credit** → fetched from UCI (direct URL, no auth)
- **Credit Card Fraud** → Kaggle: `mlg-ulb/creditcardfraud` → extracts `creditcard.csv`
- **LendingClub** → Kaggle: `wordsforthewise/lending-club` → extracts `accepted_*.csv` (~1.6 GB)

Idempotent: safe to re-run. Skips downloads that already completed, cleans up any broken partial extractions from previous attempts.

In [4]:
import os, requests, glob, gzip, shutil

def find_csv(data_dir, name_contains, min_size_mb=1):
    """Recursively find CSV files whose name contains substring (case-insensitive)
    AND are actual files (not directories) AND exceed min_size_mb."""
    matches = []
    for root, _, files in os.walk(data_dir):
        for f in files:
            full = os.path.join(root, f)
            if not os.path.isfile(full):
                continue
            if name_contains.lower() not in f.lower():
                continue
            if not f.lower().endswith('.csv'):
                continue
            if os.path.getsize(full) / 1e6 < min_size_mb:
                continue
            matches.append(full)
    return matches

def cleanup_bad_entries(data_dir, name_contains):
    """Remove directories or tiny placeholder files whose name matches the target CSV.
    These are left behind by failed/interrupted Kaggle extractions."""
    for root, dirs, files in os.walk(data_dir):
        # Remove directories named like CSVs (bug from partial extraction)
        for d in list(dirs):
            if name_contains.lower() in d.lower() and d.lower().endswith('.csv'):
                full = os.path.join(root, d)
                print(f"  Cleaning up broken directory: {full}")
                shutil.rmtree(full)
        # Remove tiny 'csv' files (<1 MB — definitely not the real one)
        for f in files:
            full = os.path.join(root, f)
            if (name_contains.lower() in f.lower()
                and f.lower().endswith('.csv')
                and os.path.isfile(full)
                and os.path.getsize(full) < 1e6):
                print(f"  Cleaning up tiny placeholder: {full}")
                os.remove(full)

def decompress_gz_files(data_dir, name_contains):
    """Decompress any matching .csv.gz into .csv and delete the .gz."""
    for root, _, files in os.walk(data_dir):
        for f in files:
            if name_contains.lower() in f.lower() and f.lower().endswith('.csv.gz'):
                full = os.path.join(root, f)
                out = full[:-3]
                if os.path.exists(out) and os.path.getsize(out) > 1e6:
                    continue
                print(f"  Decompressing {full} ...")
                with gzip.open(full, 'rb') as fi, open(out, 'wb') as fo:
                    shutil.copyfileobj(fi, fo)
                os.remove(full)

PATHS = {}

# ─── 1. German Credit from UCI ──────────────────────────────────────
print("[1/3] German Credit from UCI")
german_path = f'{DATA_DIR}/german_credit.data'
if not os.path.exists(german_path):
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(german_path, 'wb') as f:
        f.write(r.content)
PATHS['german'] = german_path
print(f"  [OK] {german_path}  ({os.path.getsize(german_path)/1024:.1f} KB)")

# ─── 2. Credit Card Fraud from Kaggle ───────────────────────────────
print("\n[2/3] Credit Card Fraud from Kaggle")
fraud_matches = find_csv(DATA_DIR, 'creditcard', min_size_mb=50)
if not fraud_matches:
    cleanup_bad_entries(DATA_DIR, 'creditcard')
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p {DATA_DIR} --unzip --quiet
    fraud_matches = find_csv(DATA_DIR, 'creditcard', min_size_mb=50)
assert fraud_matches, "creditcard.csv not found after download"
PATHS['fraud'] = fraud_matches[0]
print(f"  [OK] {PATHS['fraud']}  ({os.path.getsize(PATHS['fraud'])/1e6:.1f} MB)")

# ─── 3. LendingClub from Kaggle (slow: ~1.6 GB) ─────────────────────
print("\n[3/3] LendingClub from Kaggle (~1.6 GB, takes 2-5 min)")
lending_matches = find_csv(DATA_DIR, 'accepted', min_size_mb=100)
if not lending_matches:
    cleanup_bad_entries(DATA_DIR, 'accepted')
    !kaggle datasets download -d wordsforthewise/lending-club -p {DATA_DIR} --unzip --quiet
    decompress_gz_files(DATA_DIR, 'accepted')
    lending_matches = find_csv(DATA_DIR, 'accepted', min_size_mb=100)

assert lending_matches, "accepted_*.csv not found after LendingClub download"
PATHS['lending'] = lending_matches[0]
print(f"  [OK] {PATHS['lending']}  ({os.path.getsize(PATHS['lending'])/1e6:.1f} MB)")

print(f"\nAll datasets ready.")
for k, v in PATHS.items():
    print(f"  {k:8s}: {v}")

[1/3] German Credit from UCI
  [OK] /content/data/german_credit.data  (77.9 KB)

[2/3] Credit Card Fraud from Kaggle
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
  [OK] /content/data/creditcard.csv  (150.8 MB)

[3/3] LendingClub from Kaggle (~1.6 GB, takes 2-5 min)
Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0
  Decompressing /content/data/accepted_2007_to_2018Q4.csv.gz ...
  [OK] /content/data/accepted_2007_to_2018Q4.csv  (1675.1 MB)

All datasets ready.
  german  : /content/data/german_credit.data
  fraud   : /content/data/creditcard.csv
  lending : /content/data/accepted_2007_to_2018Q4.csv


## Cell 4 — Import Libraries & Set Seed

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import mlflow
import joblib, json, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Point MLflow at the Drive folder so runs persist across Colab sessions
mlflow.set_tracking_uri(f"file://{OUTPUT_ROOT}/mlruns")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("Libraries imported.")

MLflow tracking URI: file:///content/drive/MyDrive/CreditRisk_Wk4/mlruns
Libraries imported.


## Cell 5 — Shared Helpers

Reusable functions for stratified splitting, SMOTE application, and artifact persistence.
These are called once per dataset below.

In [6]:
def stratified_split(X, y, strat_key, val_size=0.15, test_size=0.15, seed=RANDOM_STATE):
    """
    Stratified train/val/test split.
    `strat_key` is a Series combining target + protected attribute(s) so that
    BOTH class balance AND protected-group proportions are preserved.
    Returns 6 arrays: X_train, X_val, X_test, y_train, y_val, y_test.
    """
    # First carve out test
    X_tv, X_test, y_tv, y_test, strat_tv, _ = train_test_split(
        X, y, strat_key,
        test_size=test_size,
        stratify=strat_key,
        random_state=seed,
    )
    # Then carve val out of remaining
    val_rel = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv,
        test_size=val_rel,
        stratify=strat_tv,
        random_state=seed,
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def apply_smote(X_train, y_train, seed=RANDOM_STATE):
    """
    Apply SMOTE to train set ONLY. Never touch val/test.
    Works on fully-numeric arrays (post-encoding).
    """
    before = pd.Series(y_train).value_counts().to_dict()
    smote = SMOTE(random_state=seed)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    after = pd.Series(y_res).value_counts().to_dict()
    print(f"  SMOTE: {before}  ->  {after}")
    return X_res, y_res


def save_splits(name, X_train, X_val, X_test, y_train, y_val, y_test,
                preprocessor=None, feature_names=None):
    """Save processed splits + fitted preprocessor for Week 5 modelling."""
    out = f"{OUTPUT_ROOT}/processed/{name}"
    os.makedirs(out, exist_ok=True)

    # Convert to DataFrame for parquet if we have feature names
    if feature_names is not None and not isinstance(X_train, pd.DataFrame):
        X_train = pd.DataFrame(X_train, columns=feature_names)
        X_val   = pd.DataFrame(X_val,   columns=feature_names)
        X_test  = pd.DataFrame(X_test,  columns=feature_names)

    X_train.to_parquet(f"{out}/X_train.parquet")
    X_val.to_parquet(f"{out}/X_val.parquet")
    X_test.to_parquet(f"{out}/X_test.parquet")
    pd.Series(y_train, name='target').to_frame().to_parquet(f"{out}/y_train.parquet")
    pd.Series(y_val,   name='target').to_frame().to_parquet(f"{out}/y_val.parquet")
    pd.Series(y_test,  name='target').to_frame().to_parquet(f"{out}/y_test.parquet")

    if preprocessor is not None:
        joblib.dump(preprocessor, f"{out}/preprocessor.joblib")
    print(f"  Saved -> {out}")
    return out


def log_mlflow(exp_name, run_name, params, metrics, artifact_dir):
    """Light wrapper around MLflow — logs preprocessing params + split sizes."""
    mlflow.set_experiment(exp_name)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        # Log the split artifacts (preprocessor, metadata)
        for f in ['preprocessor.joblib']:
            fp = os.path.join(artifact_dir, f)
            if os.path.exists(fp):
                mlflow.log_artifact(fp)
        print(f"  MLflow run logged under experiment '{exp_name}'")

---
# Dataset 1 — German Credit

**Target:** `credit_risk` (1 = good, 2 = bad → remapped to 0/1)
**Protected attributes (preserved in split):** `age_group`, `sex` (derived from `personal_status_sex`)
**Engineered features:** debt-to-income proxy, installment burden, employment stability


## 1.1 Load Dataset

In [ ]:
COLUMNS = [
    'checking_account', 'duration_months', 'credit_history', 'purpose',
    'credit_amount', 'savings_account', 'employment_since', 'installment_rate',
    'personal_status_sex', 'other_debtors', 'present_residence', 'property',
    'age', 'other_installments', 'housing', 'existing_credits', 'job',
    'liable_people', 'telephone', 'foreign_worker', 'credit_risk'
]

# Cell 3b fetched the raw UCI file — it's space-separated, no header
df1 = pd.read_csv(PATHS['german'], sep=' ', header=None, names=COLUMNS)

# Remap target: UCI uses 1=good, 2=bad. We want 1=bad (positive class = risk).
df1['credit_risk'] = (df1['credit_risk'] == 2).astype(int)

print(f"Loaded German Credit: {df1.shape}")
print(f"Target distribution:\n{df1['credit_risk'].value_counts()}")
display(df1.head())

Loaded German Credit: (1000, 21)
Target distribution:
credit_risk
0    700
1    300
Name: count, dtype: int64


,checking_account,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installments,housing,existing_credits,job,liable_people,telephone,foreign_worker,credit_risk
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,1


## 1.2 Handle Missing Values

German Credit has no true missing values (UCI pre-cleaned), but we run the check anyway so the pipeline is portable.

In [ ]:
miss = df1.isnull().sum()
print("Missing values per column:")
print(miss[miss > 0] if miss.sum() > 0 else "  None — dataset is clean.")

Missing values per column:
  None — dataset is clean.


## 1.3 Feature Engineering

Domain-relevant features the EDA didn't surface:

| New feature | Formula | Rationale |
|---|---|---|
| `dti_proxy` | `installment_rate × credit_amount / duration_months` | Approx monthly obligation relative to income (installment_rate is % of disposable income) |
| `credit_per_month` | `credit_amount / duration_months` | Monthly repayment size |
| `emp_stability` | Ordinal mapping of `employment_since` | Longer tenure → lower risk |
| `age_group` | Binned age (young/mid/senior) | Used as protected attribute for stratification |
| `sex` | Extracted from `personal_status_sex` | Protected attribute |


In [ ]:
df1 = df1.copy()

# Numeric engineered features
df1['dti_proxy']        = df1['installment_rate'] * df1['credit_amount'] / df1['duration_months']
df1['credit_per_month'] = df1['credit_amount'] / df1['duration_months']

# Employment stability (UCI codes: A71 unemployed ... A75 >= 7 years)
emp_map = {'A71': 0, 'A72': 1, 'A73': 2, 'A74': 3, 'A75': 4}
df1['emp_stability'] = df1['employment_since'].map(emp_map).fillna(0)

# Age group (protected attribute bin)
df1['age_group'] = pd.cut(df1['age'], bins=[0, 25, 40, 120],
                          labels=['young', 'mid', 'senior']).astype(str)

# Sex from personal_status_sex (A91-A95 encode marital+sex)
sex_map = {
    'A91': 'male', 'A92': 'female', 'A93': 'male',
    'A94': 'male', 'A95': 'female',
}
df1['sex'] = df1['personal_status_sex'].map(sex_map).fillna('male')

print("Engineered features added.")
display(df1[['dti_proxy', 'credit_per_month', 'emp_stability',
             'age_group', 'sex']].describe(include='all'))

Engineered features added.


,dti_proxy,credit_per_month,emp_stability,age_group,sex
count,1000.000000,1000.000000,1000.000000,1000,1000
unique,NaN,NaN,NaN,3,2
top,NaN,NaN,NaN,mid,male
freq,NaN,NaN,NaN,536,690
mean,430.570490,167.687020,2.384000,NaN,NaN
std,281.183505,153.490959,1.208306,NaN,NaN
min,49.708333,24.055556,0.000000,NaN,NaN
25%,251.250000,89.600000,2.000000,NaN,NaN
50%,364.386364,130.333333,2.000000,NaN,NaN
75%,522.225000,206.183333,4.000000,NaN,NaN


## 1.4 Stratified Split (preserves target + protected groups)

In [ ]:
y1 = df1['credit_risk']
X1 = df1.drop(columns=['credit_risk'])

# Composite strat key: target × age_group × sex
strat1 = df1['credit_risk'].astype(str) + '_' + df1['age_group'] + '_' + df1['sex']
print("Strat-key group sizes (smallest must be >= 2):")
print(strat1.value_counts().tail())

X1_tr, X1_va, X1_te, y1_tr, y1_va, y1_te = stratified_split(
    X1, y1, strat1, val_size=0.15, test_size=0.15
)

print(f"\nSplit sizes: train={len(X1_tr)}  val={len(X1_va)}  test={len(X1_te)}")
print("\nClass balance per split (fraction positive):")
for name, ys in [('train', y1_tr), ('val', y1_va), ('test', y1_te)]:
    print(f"  {name}: {ys.mean():.3f}")

print("\nSex proportion per split:")
for name, Xs in [('train', X1_tr), ('val', X1_va), ('test', X1_te)]:
    print(f"  {name}: {Xs['sex'].value_counts(normalize=True).to_dict()}")

Strat-key group sizes (smallest must be >= 2):
0_young_male       52
1_mid_female       49
1_young_female     47
1_young_male       33
1_senior_female    13
Name: count, dtype: int64

Split sizes: train=700  val=150  test=150

Class balance per split (fraction positive):
  train: 0.300
  val: 0.300
  test: 0.300

Sex proportion per split:
  train: {'male': 0.69, 'female': 0.31}
  val: {'male': 0.6933333333333334, 'female': 0.30666666666666664}
  test: {'male': 0.6866666666666666, 'female': 0.31333333333333335}


## 1.5 Encode & Scale (fit on train only)

`ColumnTransformer` = one-hot for categoricals, standard-scale for numerics.
Fitted on train, applied to val/test.

In [ ]:
# Drop raw columns we've replaced or don't want leaking
drop_cols = ['personal_status_sex', 'employment_since']
for X in (X1_tr, X1_va, X1_te):
    X.drop(columns=drop_cols, inplace=True, errors='ignore')

numeric_cols_1 = X1_tr.select_dtypes(include=np.number).columns.tolist()
cat_cols_1     = X1_tr.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric cols ({len(numeric_cols_1)}): {numeric_cols_1}")
print(f"Categorical cols ({len(cat_cols_1)}): {cat_cols_1}")

pre1 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), numeric_cols_1),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols_1),
])

X1_tr_enc = pre1.fit_transform(X1_tr)
X1_va_enc = pre1.transform(X1_va)
X1_te_enc = pre1.transform(X1_te)

feat_names_1 = (numeric_cols_1
                + list(pre1.named_transformers_['cat']
                       .named_steps['oh'].get_feature_names_out(cat_cols_1)))
print(f"\nEncoded shape (train): {X1_tr_enc.shape}")

Numeric cols (10): ['duration_months', 'credit_amount', 'installment_rate', 'present_residence', 'age', 'existing_credits', 'liable_people', 'dti_proxy', 'credit_per_month', 'emp_stability']
Categorical cols (13): ['checking_account', 'credit_history', 'purpose', 'savings_account', 'other_debtors', 'property', 'other_installments', 'housing', 'job', 'telephone', 'foreign_worker', 'age_group', 'sex']

Encoded shape (train): (700, 60)


## 1.6 SMOTE on Train Only

In [ ]:
X1_tr_bal, y1_tr_bal = apply_smote(X1_tr_enc, y1_tr)

  SMOTE: {0: 490, 1: 210}  ->  {0: 490, 1: 490}


## 1.7 Save Splits & Log to MLflow

In [ ]:
out1 = save_splits(
    'german', X1_tr_bal, X1_va_enc, X1_te_enc,
    y1_tr_bal, y1_va, y1_te,
    preprocessor=pre1, feature_names=feat_names_1,
)

log_mlflow(
    exp_name='wk4_preprocessing',
    run_name='german_credit',
    params={
        'dataset': 'german_credit',
        'n_features_raw': X1.shape[1],
        'n_features_encoded': X1_tr_enc.shape[1],
        'smote': True,
        'val_size': 0.15,
        'test_size': 0.15,
        'protected_attrs': 'age_group,sex',
    },
    metrics={
        'n_train_after_smote': len(y1_tr_bal),
        'n_val': len(y1_va),
        'n_test': len(y1_te),
        'pos_rate_train_raw': float(y1_tr.mean()),
        'pos_rate_val':       float(y1_va.mean()),
        'pos_rate_test':      float(y1_te.mean()),
    },
    artifact_dir=out1,
)

  Saved -> /content/drive/MyDrive/CreditRisk_Wk4/processed/german
  MLflow run logged under experiment 'wk4_preprocessing'


---
# Dataset 2 — LendingClub

**Target:** `loan_status` → binary (Charged Off / Default = 1, Fully Paid = 0)
**Protected attributes (preserved in split):** `addr_state_region` (Census region), `emp_length_group`
**Engineered features:** debt-to-income (already present as `dti`), credit utilisation, employment stability, income-to-loan ratio


## 2.1 Load Dataset

In [7]:
df2 = pd.read_csv(PATHS['lending'], low_memory=False) if PATHS['lending'] else None
assert df2 is not None, "LendingClub CSV not found — upload it to Drive."
print(f"Loaded LendingClub: {df2.shape}")
print(f"Columns (first 20): {df2.columns.tolist()[:20]}")

Loaded LendingClub: (2260701, 151)
Columns (first 20): ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc']


## 2.2 Target + Basic Filtering

Keep only completed loans (`Fully Paid` vs `Charged Off`/`Default`). In-progress loans are ambiguous labels.

In [8]:
# Different LC exports use different column names — handle both
status_col = 'loan_status' if 'loan_status' in df2.columns else None
assert status_col is not None, f"No loan_status column. Got: {df2.columns.tolist()[:10]}"

keep = ['Fully Paid', 'Charged Off', 'Default']
df2 = df2[df2[status_col].isin(keep)].copy()
df2['target'] = (df2[status_col].isin(['Charged Off', 'Default'])).astype(int)
df2 = df2.drop(columns=[status_col])

print(f"After filtering: {df2.shape}")
print(f"Target (1=default):\n{df2['target'].value_counts()}")
print(f"Default rate: {df2['target'].mean():.3f}")

After filtering: (1345350, 151)
Target (1=default):
target
0    1076751
1     268599
Name: count, dtype: int64
Default rate: 0.200


## 2.3 Select Candidate Features

LendingClub has 100+ columns, many post-origination (data leakage). We keep only features known **at loan origination**.

In [9]:
# Origination-time features only (no post-hoc fields like total_pymnt, recoveries, etc.)
candidate_features = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies',
]
keep_cols = [c for c in candidate_features if c in df2.columns] + ['target']
df2 = df2[keep_cols].copy()
print(f"Kept {len(keep_cols)-1} features: {keep_cols[:-1]}")

Kept 24 features: ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies']


## 2.4 Clean Column Formats

LendingClub CSVs store some numeric columns as strings with units (`' 36 months'`, `'10.99%'`, `'< 1 year'`).

In [10]:
def clean_term(x):
    if pd.isna(x): return np.nan
    return int(str(x).strip().split()[0])

def clean_pct(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().rstrip('%')
    try: return float(s)
    except: return np.nan

def clean_emp_length(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if '<' in s: return 0
    if '10' in s: return 10
    # '1 year', '2 years' ...
    for tok in s.split():
        if tok.isdigit(): return int(tok)
    return np.nan

if 'term' in df2.columns:        df2['term']        = df2['term'].apply(clean_term)
if 'int_rate' in df2.columns:    df2['int_rate']    = df2['int_rate'].apply(clean_pct)
if 'revol_util' in df2.columns:  df2['revol_util']  = df2['revol_util'].apply(clean_pct)
if 'emp_length' in df2.columns:  df2['emp_length']  = df2['emp_length'].apply(clean_emp_length)

print("Cleaned numeric strings.")
display(df2.dtypes.value_counts())

Cleaned numeric strings.


,count
float64,15
object,8
int64,2


## 2.5 Handle Missing Values

In [11]:
miss = df2.isnull().sum().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"Columns with missing values: {len(miss)}")
display(miss.head(15))

# Drop columns >50% missing (if any), impute the rest downstream in the ColumnTransformer
high_miss = miss[miss > 0.5 * len(df2)].index.tolist()
if high_miss:
    print(f"\nDropping {len(high_miss)} cols with >50% missing: {high_miss}")
    df2 = df2.drop(columns=high_miss)

Columns with missing values: 6


,0
emp_length,78516
mort_acc,47281
revol_util,857
pub_rec_bankruptcies,697
dti,374
inq_last_6mths,1


## 2.6 Feature Engineering

| New feature | Formula | Rationale |
|---|---|---|
| `inc_to_loan` | `annual_inc / loan_amnt` | Affordability ratio |
| `installment_to_inc` | `installment * 12 / annual_inc` | Annual burden share |
| `credit_utilisation_bucket` | binned `revol_util` | Standard credit-risk feature |
| `emp_length_group` | short/mid/long | Protected attribute for stratification |
| `addr_state_region` | state → Census region | Protected attribute (reduces cardinality) |


In [12]:
df2 = df2.copy()

# Affordability
if {'annual_inc', 'loan_amnt'}.issubset(df2.columns):
    df2['inc_to_loan'] = df2['annual_inc'] / df2['loan_amnt'].replace(0, np.nan)
if {'installment', 'annual_inc'}.issubset(df2.columns):
    df2['installment_to_inc'] = df2['installment'] * 12 / df2['annual_inc'].replace(0, np.nan)

# Credit utilisation bucket
if 'revol_util' in df2.columns:
    df2['credit_utilisation_bucket'] = pd.cut(
        df2['revol_util'], bins=[-0.1, 30, 60, 90, 1e6],
        labels=['low', 'med', 'high', 'maxed']).astype(str)

# Employment stability (protected attribute)
if 'emp_length' in df2.columns:
    df2['emp_length_group'] = pd.cut(
        df2['emp_length'].fillna(-1),
        bins=[-2, 0, 3, 7, 11],
        labels=['missing', 'short', 'mid', 'long']).astype(str)

# State -> Census region
CENSUS_REGION = {
    # Northeast
    **dict.fromkeys(['CT','ME','MA','NH','RI','VT','NJ','NY','PA'], 'northeast'),
    # Midwest
    **dict.fromkeys(['IL','IN','MI','OH','WI','IA','KS','MN','MO','NE','ND','SD'], 'midwest'),
    # South
    **dict.fromkeys(['DE','FL','GA','MD','NC','SC','VA','DC','WV',
                     'AL','KY','MS','TN','AR','LA','OK','TX'], 'south'),
    # West
    **dict.fromkeys(['AZ','CO','ID','MT','NV','NM','UT','WY',
                     'AK','CA','HI','OR','WA'], 'west'),
}
if 'addr_state' in df2.columns:
    df2['addr_state_region'] = df2['addr_state'].map(CENSUS_REGION).fillna('other')
    df2 = df2.drop(columns=['addr_state'])  # keep region, drop 50-way cardinal column

print("Engineered features added.")
display(df2[[c for c in ['inc_to_loan','installment_to_inc',
                         'credit_utilisation_bucket','emp_length_group',
                         'addr_state_region'] if c in df2.columns]].describe(include='all'))

Engineered features added.


,inc_to_loan,installment_to_inc,credit_utilisation_bucket,emp_length_group,addr_state_region
count,1.345350e+06,1.344989e+06,1345350,1345350,1345350
unique,NaN,NaN,5,4,4
top,NaN,NaN,med,long,south
freq,NaN,NaN,538167,553850,478594
mean,7.285484e+00,1.423694e-01,NaN,NaN,NaN
std,1.145861e+01,2.364355e+01,NaN,NaN,NaN
min,0.000000e+00,6.912000e-05,NaN,NaN,NaN
25%,3.437500e+00,4.626102e-02,NaN,NaN,NaN
50%,5.000000e+00,7.219950e-02,NaN,NaN,NaN
75%,8.019987e+00,1.054020e-01,NaN,NaN,NaN


## 2.7 Optional Downsample (Colab memory)

LendingClub can be 1M+ rows. For Colab free-tier memory we downsample **stratified on target** to keep class ratio intact. Set `SAMPLE_N = None` to skip.

In [13]:
SAMPLE_N = 150_000   # set to None to use full dataset

if SAMPLE_N and len(df2) > SAMPLE_N:
    df2, _ = train_test_split(
        df2, train_size=SAMPLE_N,
        stratify=df2['target'],
        random_state=RANDOM_STATE,
    )
    df2 = df2.reset_index(drop=True)
    print(f"Downsampled to {len(df2)} rows  (target mean preserved: {df2['target'].mean():.3f})")
else:
    print(f"Using full dataset: {len(df2)} rows")

Downsampled to 150000 rows  (target mean preserved: 0.200)


## 2.8 Stratified Split

In [14]:
y2 = df2['target']
X2 = df2.drop(columns=['target'])

# Composite strat on target × region × emp_length_group
strat_parts = [df2['target'].astype(str)]
if 'addr_state_region' in df2.columns:
    strat_parts.append(df2['addr_state_region'])
if 'emp_length_group' in df2.columns:
    strat_parts.append(df2['emp_length_group'])
strat2 = strat_parts[0].str.cat(strat_parts[1:], sep='_')

# Some strat groups may have fewer than 2 members — fall back to target+region only
if strat2.value_counts().min() < 2:
    strat2 = df2['target'].astype(str) + '_' + df2.get('addr_state_region', 'x').astype(str)
    print("Fell back to target × region stratification (some groups too small).")

X2_tr, X2_va, X2_te, y2_tr, y2_va, y2_te = stratified_split(
    X2, y2, strat2, val_size=0.15, test_size=0.15
)

print(f"\nSplit sizes: train={len(X2_tr)}  val={len(X2_va)}  test={len(X2_te)}")
for name, ys in [('train', y2_tr), ('val', y2_va), ('test', y2_te)]:
    print(f"  {name} pos rate: {ys.mean():.3f}")

if 'addr_state_region' in X2_tr.columns:
    print("\nRegion proportions per split:")
    for name, Xs in [('train', X2_tr), ('val', X2_va), ('test', X2_te)]:
        print(f"  {name}: {Xs['addr_state_region'].value_counts(normalize=True).round(3).to_dict()}")


Split sizes: train=105000  val=22500  test=22500
  train pos rate: 0.200
  val pos rate: 0.200
  test pos rate: 0.200

Region proportions per split:
  train: {'south': 0.357, 'west': 0.268, 'northeast': 0.201, 'midwest': 0.174}
  val: {'south': 0.357, 'west': 0.268, 'northeast': 0.201, 'midwest': 0.174}
  test: {'south': 0.357, 'west': 0.268, 'northeast': 0.201, 'midwest': 0.174}


## 2.9 Encode & Scale

In [15]:
numeric_cols_2 = X2_tr.select_dtypes(include=np.number).columns.tolist()
cat_cols_2     = X2_tr.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric ({len(numeric_cols_2)}): {numeric_cols_2}")
print(f"Categorical ({len(cat_cols_2)}): {cat_cols_2}")

pre2 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), numeric_cols_2),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols_2),
])

X2_tr_enc = pre2.fit_transform(X2_tr)
X2_va_enc = pre2.transform(X2_va)
X2_te_enc = pre2.transform(X2_te)

feat_names_2 = (numeric_cols_2
                + list(pre2.named_transformers_['cat']
                       .named_steps['oh'].get_feature_names_out(cat_cols_2)))
print(f"\nEncoded shape (train): {X2_tr_enc.shape}")

Numeric (18): ['loan_amnt', 'term', 'int_rate', 'installment', 'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'inc_to_loan', 'installment_to_inc']
Categorical (10): ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type', 'credit_utilisation_bucket', 'emp_length_group', 'addr_state_region']

Encoded shape (train): (105000, 100)


## 2.10 SMOTE on Train Only

In [16]:
X2_tr_bal, y2_tr_bal = apply_smote(X2_tr_enc, y2_tr)

  SMOTE: {0: 84038, 1: 20962}  ->  {1: 84038, 0: 84038}


## 2.11 Save Splits & Log to MLflow

In [17]:
out2 = save_splits(
    'lendingclub', X2_tr_bal, X2_va_enc, X2_te_enc,
    y2_tr_bal, y2_va, y2_te,
    preprocessor=pre2, feature_names=feat_names_2,
)

log_mlflow(
    exp_name='wk4_preprocessing',
    run_name='lendingclub',
    params={
        'dataset': 'lendingclub',
        'n_features_raw': X2.shape[1],
        'n_features_encoded': X2_tr_enc.shape[1],
        'smote': True,
        'val_size': 0.15,
        'test_size': 0.15,
        'protected_attrs': 'addr_state_region,emp_length_group',
        'sampled_rows': len(df2),
    },
    metrics={
        'n_train_after_smote': len(y2_tr_bal),
        'n_val': len(y2_va),
        'n_test': len(y2_te),
        'pos_rate_train_raw': float(y2_tr.mean()),
        'pos_rate_val':       float(y2_va.mean()),
        'pos_rate_test':      float(y2_te.mean()),
    },
    artifact_dir=out2,
)

  Saved -> /content/drive/MyDrive/CreditRisk_Wk4/processed/lendingclub
  MLflow run logged under experiment 'wk4_preprocessing'


---
# Dataset 3 — Credit Card Fraud Detection

**Target:** `Class` (1 = fraud, 0.173% positive rate — severely imbalanced)
**Protected attributes:** *none* — features V1-V28 are anonymized PCA components; `Time` and `Amount` aren't demographic.
We stratify on **target only**.
**Engineered features:** `log_amount`, `hour_of_day`, `amount_bucket`

⚠️ SMOTE at this imbalance level multiplies the minority class ~580x. This is standard practice for this dataset but the effective `k_neighbors` must be modest. We keep default `k=5`; consider `BorderlineSMOTE` in Week 5 if models overfit synthetic points.


## 3.1 Load Dataset

In [ ]:
df3 = pd.read_csv(PATHS['fraud']) if PATHS['fraud'] else None
assert df3 is not None, "creditcard.csv not found — upload to Drive."
print(f"Loaded Fraud dataset: {df3.shape}")
print(f"Class distribution:\n{df3['Class'].value_counts()}")
print(f"Fraud rate: {df3['Class'].mean()*100:.3f}%")

Loaded Fraud dataset: (284807, 31)
Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64
Fraud rate: 0.173%


## 3.2 Missing Values

In [ ]:
miss = df3.isnull().sum().sum()
print(f"Total missing values: {miss}")

Total missing values: 0


## 3.3 Feature Engineering

V1–V28 are already PCA-transformed so we leave them as-is. `Amount` and `Time` are raw.

In [ ]:
df3 = df3.copy()

# log(1+amount) — amount is heavy-tailed (EDA showed max $25k)
df3['log_amount'] = np.log1p(df3['Amount'])

# Time -> hour of day (Time is seconds elapsed from first transaction, 48h span)
df3['hour_of_day'] = ((df3['Time'] / 3600) % 24).astype(int)

# Amount buckets
df3['amount_bucket'] = pd.cut(
    df3['Amount'], bins=[-0.01, 10, 100, 1000, 1e9],
    labels=['micro', 'small', 'medium', 'large']).astype(str)

print("Engineered features added.")
display(df3[['log_amount', 'hour_of_day', 'amount_bucket']].describe(include='all'))

Engineered features added.


,log_amount,hour_of_day,amount_bucket
count,284807.000000,284807.000000,284807
unique,NaN,NaN,4
top,NaN,NaN,small
freq,NaN,NaN,128035
mean,3.152188,14.046470,NaN
std,1.656648,5.835854,NaN
min,0.000000,0.000000,NaN
25%,1.887070,10.000000,NaN
50%,3.135494,15.000000,NaN
75%,4.358822,19.000000,NaN


## 3.4 Stratified Split (target only)

In [ ]:
y3 = df3['Class']
X3 = df3.drop(columns=['Class'])

X3_tr, X3_va, X3_te, y3_tr, y3_va, y3_te = stratified_split(
    X3, y3, strat_key=y3, val_size=0.15, test_size=0.15
)

print(f"Split sizes: train={len(X3_tr)}  val={len(X3_va)}  test={len(X3_te)}")
for name, ys in [('train', y3_tr), ('val', y3_va), ('test', y3_te)]:
    print(f"  {name}: {ys.sum()} fraud of {len(ys)}  ({ys.mean()*100:.3f}%)")

Split sizes: train=199364  val=42721  test=42722
  train: 344 fraud of 199364  (0.173%)
  val: 74 fraud of 42721  (0.173%)
  test: 74 fraud of 42722  (0.173%)


## 3.5 Encode & Scale

In [ ]:
numeric_cols_3 = X3_tr.select_dtypes(include=np.number).columns.tolist()
cat_cols_3     = X3_tr.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric ({len(numeric_cols_3)}): {numeric_cols_3[:8]}...")
print(f"Categorical ({len(cat_cols_3)}): {cat_cols_3}")

pre3 = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols_3),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_3),
])

X3_tr_enc = pre3.fit_transform(X3_tr)
X3_va_enc = pre3.transform(X3_va)
X3_te_enc = pre3.transform(X3_te)

feat_names_3 = (numeric_cols_3
                + list(pre3.named_transformers_['cat']
                       .get_feature_names_out(cat_cols_3)))
print(f"\nEncoded shape (train): {X3_tr_enc.shape}")

Numeric (32): ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7']...
Categorical (1): ['amount_bucket']

Encoded shape (train): (199364, 36)


## 3.6 SMOTE on Train Only

In [ ]:
X3_tr_bal, y3_tr_bal = apply_smote(X3_tr_enc, y3_tr)

  SMOTE: {0: 199020, 1: 344}  ->  {0: 199020, 1: 199020}


## 3.7 Save Splits & Log to MLflow

In [ ]:
out3 = save_splits(
    'fraud', X3_tr_bal, X3_va_enc, X3_te_enc,
    y3_tr_bal, y3_va, y3_te,
    preprocessor=pre3, feature_names=feat_names_3,
)

log_mlflow(
    exp_name='wk4_preprocessing',
    run_name='credit_card_fraud',
    params={
        'dataset': 'credit_card_fraud',
        'n_features_raw': X3.shape[1],
        'n_features_encoded': X3_tr_enc.shape[1],
        'smote': True,
        'val_size': 0.15,
        'test_size': 0.15,
        'protected_attrs': 'none',
    },
    metrics={
        'n_train_after_smote': len(y3_tr_bal),
        'n_val': len(y3_va),
        'n_test': len(y3_te),
        'pos_rate_train_raw': float(y3_tr.mean()),
        'pos_rate_val':       float(y3_va.mean()),
        'pos_rate_test':      float(y3_te.mean()),
    },
    artifact_dir=out3,
)

  Saved -> /content/drive/MyDrive/CreditRisk_Wk4/processed/fraud
  MLflow run logged under experiment 'wk4_preprocessing'


---
# Environment Setup — Docker + MLflow

For reproducibility across machines, we write a `Dockerfile` and `requirements.txt` to the project folder.

## 4.1 Write `requirements.txt`

In [ ]:
requirements = """numpy==1.26.4
pandas==2.2.2
scikit-learn==1.5.1
imbalanced-learn==0.12.3
matplotlib==3.8.4
seaborn==0.13.2
mlflow==2.16.2
joblib==1.4.2
pyarrow==16.1.0
jupyterlab==4.2.5
"""

req_path = f'{OUTPUT_ROOT}/requirements.txt'
with open(req_path, 'w') as f:
    f.write(requirements)
print(f"Wrote {req_path}")
print(requirements)

Wrote /content/drive/MyDrive/CreditRisk_Wk4/requirements.txt
numpy==1.26.4
pandas==2.2.2
scikit-learn==1.5.1
imbalanced-learn==0.12.3
matplotlib==3.8.4
seaborn==0.13.2
mlflow==2.16.2
joblib==1.4.2
pyarrow==16.1.0
jupyterlab==4.2.5



## 4.2 Write `Dockerfile`

Builds a reproducible image with the exact library versions above. Run locally with:

```bash
docker build -t credit-risk-wk4 .
docker run -p 8888:8888 -p 5000:5000 -v $(pwd):/workspace credit-risk-wk4
```


In [ ]:
dockerfile = """FROM python:3.10-slim

WORKDIR /workspace

# System deps for pandas/pyarrow/matplotlib
RUN apt-get update && apt-get install -y --no-install-recommends \
        build-essential git && \
    rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

EXPOSE 8888 5000

# Default: launch Jupyter Lab. Override with `docker run ... mlflow ui` for tracking UI.
CMD ["jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root"]
"""

docker_path = f'{OUTPUT_ROOT}/Dockerfile'
with open(docker_path, 'w') as f:
    f.write(dockerfile)
print(f"Wrote {docker_path}")

Wrote /content/drive/MyDrive/CreditRisk_Wk4/Dockerfile


## 4.3 Launch MLflow UI (inside Colab)

This starts the MLflow server on port 5000 and exposes it via ngrok so you can view runs from your browser.
**Optional** — skip if you'll view MLflow from your local Docker later by reading the same Drive folder.

In [ ]:
# Uncomment to launch inside Colab (requires ngrok auth token)
#
# !pip install -q pyngrok
# from pyngrok import ngrok
# # ngrok.set_auth_token("YOUR_NGROK_TOKEN")
# get_ipython().system_raw(f"mlflow ui --backend-store-uri file://{OUTPUT_ROOT}/mlruns --port 5000 &")
# public_url = ngrok.connect(5000)
# print(f"MLflow UI: {public_url}")

print("To view MLflow runs later (locally):")
print(f"  mlflow ui --backend-store-uri file://{OUTPUT_ROOT}/mlruns")

To view MLflow runs later (locally):
  mlflow ui --backend-store-uri file:///content/drive/MyDrive/CreditRisk_Wk4/mlruns


## 4.4 Verify All Outputs

In [ ]:
import os
print(f"Contents of {OUTPUT_ROOT}:\n")
for root, dirs, files in os.walk(OUTPUT_ROOT):
    depth = root.replace(OUTPUT_ROOT, '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{indent}  {f}  ({size:.1f} KB)")

Contents of /content/drive/MyDrive/CreditRisk_Wk4:

CreditRisk_Wk4/
  requirements.txt  (0.2 KB)
  Dockerfile  (0.5 KB)
  processed/
    german/
      X_train.parquet  (119.6 KB)
      X_val.parquet  (49.6 KB)
      X_test.parquet  (49.5 KB)
      y_train.parquet  (1.8 KB)
      y_val.parquet  (3.0 KB)
      y_test.parquet  (3.0 KB)
      preprocessor.joblib  (7.2 KB)
    lendingclub/
      X_train.parquet  (18241.9 KB)
      X_val.parquet  (1121.7 KB)
      X_test.parquet  (1121.9 KB)
      y_train.parquet  (17.1 KB)
      y_val.parquet  (149.0 KB)
      y_test.parquet  (148.8 KB)
      preprocessor.joblib  (7.4 KB)
    fraud/
      X_train.parquet  (105898.6 KB)
      X_val.parquet  (12357.3 KB)
      X_test.parquet  (12356.0 KB)
      y_train.parquet  (3.0 KB)
      y_val.parquet  (286.3 KB)
      y_test.parquet  (286.2 KB)
      preprocessor.joblib  (3.9 KB)
  mlruns/
    .trash/
    665290085186986806/
      meta.yaml  (0.2 KB)
      cfc04cbb312c4c9d99ba749fc319ee85/
        meta.

---
# Summary

| Dataset | Raw features | Encoded features | Train (post-SMOTE) | Val | Test | Protected attrs |
|---|---|---|---|---|---|---|
| German Credit | ~20 | see 1.5 | balanced | 15% | 15% | age_group, sex |
| LendingClub | ~24 | see 2.9 | balanced | 15% | 15% | addr_state_region, emp_length_group |
| Credit Card Fraud | 30 | see 3.5 | balanced | 15% | 15% | none (anonymized) |

**Saved to Drive under `CreditRisk_Wk4/processed/<dataset>/`:**
- `X_train.parquet`, `X_val.parquet`, `X_test.parquet`
- `y_train.parquet`, `y_val.parquet`, `y_test.parquet`
- `preprocessor.joblib` — fitted ColumnTransformer (needed for inference in Week 5)

**MLflow experiment:** `wk4_preprocessing` — 3 runs, one per dataset.

**Next week:** load these parquet files, train XGBoost / LogReg / RandomForest, compare via MLflow.
